# 68 - Checkmate Signal + Our Strategies

**Goal:** Test if the Checkmate composite signal improves our existing strategies:

| Strategy | Entry | Exit | Return |
|----------|-------|------|--------|
| STRAT-002 | SOPR<1 + STH-SOPR<1 + RL Z>0.5 | 30% trail | +5,754% |
| STRAT-003 | STH-SOPR<1 | LTH-SOPR>1.5 triggered trail | +3,813% |

**Ways to use Checkmate signal:**
1. **Entry confirmation** - Only enter if signal agrees
2. **Exit confirmation** - Only exit if signal agrees  
3. **Position sizing** - Size based on signal level
4. **Trail adjustment** - Tighter/looser trail based on signal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        print(f"  Warning: {name} not found")
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

# Load all needed metrics
metrics = ['price', 'sopr', 'sopr_sth', 'sopr_lth', 'mvrv', 'mvrv_sth', 'mvrv_lth', 
           'nupl', 'aviv', 'realized_loss']
data = {m: load_metric(m) for m in metrics}
print("Data loaded")

In [ ]:
# Build master dataframe
df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

# Add all metrics
metric_map = {
    'sopr': 'sopr',
    'sopr_sth': 'sth_sopr',
    'sopr_lth': 'lth_sopr',
    'mvrv': 'mvrv',
    'mvrv_sth': 'mvrv_sth',
    'mvrv_lth': 'mvrv_lth',
    'nupl': 'nupl',
    'aviv': 'aviv',
    'realized_loss': 'realized_loss'
}

for src, dst in metric_map.items():
    if src in data and not data[src].empty:
        df = df.join(data[src][['value']].rename(columns={'value': dst}), how='left')
        df[dst] = df[dst].ffill()

# Calculate Realized Loss Z-Score
if 'realized_loss' in df.columns:
    df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(365).mean()) / df['realized_loss'].rolling(365).std()
else:
    df['rl_zscore'] = 0

print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Build Checkmate composite signal
SIGNAL_CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

def calc_checkmate_signal(row):
    total_score, total_weight = 0, 0
    for metric, cfg in SIGNAL_CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['checkmate_signal'] = df.apply(calc_checkmate_signal, axis=1)
print(f"Checkmate signal range: {df['checkmate_signal'].min():.2f} to {df['checkmate_signal'].max():.2f}")

---
## Part 1: Define Our Strategies

In [ ]:
# STRAT-002 Entry: SOPR<1 + STH-SOPR<1 + RL Z>0.5
df['strat002_entry_raw'] = (
    (df['sopr'] < 1.0) & 
    (df['sth_sopr'] < 1.0) & 
    (df['rl_zscore'] > 0.5)
)
# First day only (not continuation)
df['strat002_entry'] = df['strat002_entry_raw'] & ~df['strat002_entry_raw'].shift(1).fillna(False)

# STRAT-003 Entry: STH-SOPR<1
df['strat003_entry_raw'] = df['sth_sopr'] < 1.0
df['strat003_entry'] = df['strat003_entry_raw'] & ~df['strat003_entry_raw'].shift(1).fillna(False)

# STRAT-003 Exit trigger: MVRV>2.5 + LTH-SOPR>1.5
df['strat003_exit_trigger'] = (df['mvrv'] > 2.5) & (df['lth_sopr'] > 1.5)

print("Entry signals:")
print(f"  STRAT-002 entries: {df['strat002_entry'].sum()}")
print(f"  STRAT-003 entries: {df['strat003_entry'].sum()}")
print(f"  STRAT-003 exit triggers: {df['strat003_exit_trigger'].sum()} days")

In [ ]:
def backtest_strategy(df, entry_col, exit_mode='trail', trail_pct=0.30, 
                       trigger_col=None, tight_trail=0.15,
                       entry_confirm=None, exit_confirm=None,
                       position_sizing=None):
    """
    Backtest a strategy with optional Checkmate signal confirmation.
    
    Args:
        entry_col: Column with entry signals
        exit_mode: 'trail' or 'trigger_trail'
        trail_pct: Trailing stop percentage
        trigger_col: Column with exit trigger (for trigger_trail mode)
        tight_trail: Tighter trail after trigger
        entry_confirm: Checkmate signal level for entry confirmation (< this)
        exit_confirm: Checkmate signal level for exit confirmation (> this)
        position_sizing: 'signal' to use signal-based sizing
    """
    bt = df[['price', 'returns', 'checkmate_signal', entry_col]].copy()
    if trigger_col:
        bt['trigger'] = df[trigger_col]
    
    # Track trades
    in_position = False
    entry_price = 0
    peak_price = 0
    triggered = False
    position_size = 1.0
    positions = []
    trades = []
    
    for date, row in bt.iterrows():
        price = row['price']
        signal = row['checkmate_signal']
        
        if not in_position:
            # Check for entry
            entry_signal = row[entry_col]
            
            # Apply entry confirmation
            if entry_confirm is not None:
                entry_signal = entry_signal and (signal < entry_confirm)
            
            if entry_signal:
                in_position = True
                entry_price = price
                peak_price = price
                triggered = False
                
                # Position sizing
                if position_sizing == 'signal':
                    if signal <= -1.0: position_size = 1.0
                    elif signal <= -0.5: position_size = 0.8
                    elif signal <= 0.0: position_size = 0.6
                    elif signal <= 0.5: position_size = 0.4
                    else: position_size = 0.25
                else:
                    position_size = 1.0
                
                trades.append({'date': date, 'action': 'BUY', 'price': price, 'signal': signal, 'size': position_size})
            
            positions.append(0)
        
        else:
            # Update peak
            peak_price = max(peak_price, price)
            
            # Check for trigger
            if trigger_col and not triggered:
                if row.get('trigger', False):
                    triggered = True
            
            # Determine trail percentage
            current_trail = tight_trail if triggered else trail_pct
            
            # Check exit
            drawdown = (peak_price - price) / peak_price
            exit_signal = drawdown >= current_trail
            
            # Apply exit confirmation (optional)
            if exit_confirm is not None and exit_signal:
                exit_signal = exit_signal and (signal > exit_confirm)
            
            if exit_signal:
                in_position = False
                pnl = (price / entry_price - 1) * 100
                trades.append({'date': date, 'action': 'SELL', 'price': price, 'signal': signal, 
                              'pnl': pnl, 'triggered': triggered})
                positions.append(0)
            else:
                positions.append(position_size)
    
    bt['position'] = positions
    bt['position'] = bt['position'].shift(1).fillna(0)
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    sells = [t for t in trades if t['action'] == 'SELL']
    
    return {
        'total_return': (bt['equity'].iloc[-1] / 100000 - 1) * 100,
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'trades': len(sells),
        'win_rate': len([t for t in sells if t['pnl'] > 0]) / len(sells) * 100 if sells else 0,
        'avg_pnl': np.mean([t['pnl'] for t in sells]) if sells else 0,
        'equity': bt['equity'],
        'dd': bt['dd'],
        'trade_log': trades
    }

---
## Part 2: Test STRAT-002 with Checkmate Confirmation

In [ ]:
print("="*100)
print("STRAT-002: SOPR + STH-SOPR + RL Z → 30% Trail")
print("="*100)

# Baseline
baseline = backtest_strategy(df, 'strat002_entry', exit_mode='trail', trail_pct=0.30)

# With entry confirmations
configs = [
    {'name': 'Baseline (no confirm)', 'entry_confirm': None, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < 0.5', 'entry_confirm': 0.5, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < 0', 'entry_confirm': 0.0, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < -0.5', 'entry_confirm': -0.5, 'exit_confirm': None, 'sizing': None},
    {'name': 'Signal-based sizing', 'entry_confirm': None, 'exit_confirm': None, 'sizing': 'signal'},
    {'name': 'Entry<0 + Sizing', 'entry_confirm': 0.0, 'exit_confirm': None, 'sizing': 'signal'},
]

results_002 = []
for cfg in configs:
    result = backtest_strategy(df, 'strat002_entry', exit_mode='trail', trail_pct=0.30,
                                entry_confirm=cfg['entry_confirm'],
                                exit_confirm=cfg['exit_confirm'],
                                position_sizing=cfg['sizing'])
    result['name'] = cfg['name']
    results_002.append(result)

print(f"\n{'Config':<25} {'Return':>12} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Trades':>8} {'WinRate':>8}")
print("-"*90)
for r in results_002:
    print(f"{r['name']:<25} {r['total_return']:>11.0f}% {r['cagr']:>7.1f}% {r['max_dd']:>7.1f}% "
          f"{r['sharpe']:>8.2f} {r['trades']:>8} {r['win_rate']:>7.0f}%")

---
## Part 3: Test STRAT-003 with Checkmate Confirmation

In [ ]:
print("\n" + "="*100)
print("STRAT-003: STH-SOPR<1 → LTH-SOPR Triggered Trail")
print("="*100)

configs = [
    {'name': 'Baseline (no confirm)', 'entry_confirm': None, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < 0.5', 'entry_confirm': 0.5, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < 0', 'entry_confirm': 0.0, 'exit_confirm': None, 'sizing': None},
    {'name': 'Entry: signal < -0.5', 'entry_confirm': -0.5, 'exit_confirm': None, 'sizing': None},
    {'name': 'Signal-based sizing', 'entry_confirm': None, 'exit_confirm': None, 'sizing': 'signal'},
    {'name': 'Entry<0 + Sizing', 'entry_confirm': 0.0, 'exit_confirm': None, 'sizing': 'signal'},
]

results_003 = []
for cfg in configs:
    result = backtest_strategy(df, 'strat003_entry', exit_mode='trigger_trail', 
                                trail_pct=0.30, trigger_col='strat003_exit_trigger', tight_trail=0.15,
                                entry_confirm=cfg['entry_confirm'],
                                exit_confirm=cfg['exit_confirm'],
                                position_sizing=cfg['sizing'])
    result['name'] = cfg['name']
    results_003.append(result)

print(f"\n{'Config':<25} {'Return':>12} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Trades':>8} {'WinRate':>8}")
print("-"*90)
for r in results_003:
    print(f"{r['name']:<25} {r['total_return']:>11.0f}% {r['cagr']:>7.1f}% {r['max_dd']:>7.1f}% "
          f"{r['sharpe']:>8.2f} {r['trades']:>8} {r['win_rate']:>7.0f}%")

---
## Part 4: Signal-Based Trail Adjustment

In [ ]:
def backtest_adaptive_trail(df, entry_col, base_trail=0.30):
    """
    Trail stop tightens when Checkmate signal turns bearish.
    - Signal < 0: Use base trail (30%)
    - Signal 0-0.5: Tighten to 25%
    - Signal 0.5-1.0: Tighten to 20%
    - Signal > 1.0: Tighten to 15%
    """
    bt = df[['price', 'returns', 'checkmate_signal', entry_col]].copy()
    
    in_position = False
    entry_price = 0
    peak_price = 0
    positions = []
    trades = []
    
    for date, row in bt.iterrows():
        price = row['price']
        signal = row['checkmate_signal']
        
        if not in_position:
            if row[entry_col]:
                in_position = True
                entry_price = price
                peak_price = price
                trades.append({'date': date, 'action': 'BUY', 'price': price, 'signal': signal})
            positions.append(0)
        else:
            peak_price = max(peak_price, price)
            
            # Adaptive trail based on signal
            if signal < 0:
                trail_pct = base_trail  # 30% - bullish, give room
            elif signal < 0.5:
                trail_pct = 0.25  # Neutral
            elif signal < 1.0:
                trail_pct = 0.20  # Getting bearish
            else:
                trail_pct = 0.15  # Bearish - tight trail
            
            drawdown = (peak_price - price) / peak_price
            
            if drawdown >= trail_pct:
                in_position = False
                pnl = (price / entry_price - 1) * 100
                trades.append({'date': date, 'action': 'SELL', 'price': price, 'signal': signal, 
                              'pnl': pnl, 'trail_used': trail_pct})
                positions.append(0)
            else:
                positions.append(1)
    
    bt['position'] = positions
    bt['position'] = bt['position'].shift(1).fillna(0)
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    sells = [t for t in trades if t['action'] == 'SELL']
    
    return {
        'total_return': (bt['equity'].iloc[-1] / 100000 - 1) * 100,
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'trades': len(sells),
        'win_rate': len([t for t in sells if t['pnl'] > 0]) / len(sells) * 100 if sells else 0,
        'equity': bt['equity'],
        'trade_log': trades
    }

print("\n" + "="*100)
print("ADAPTIVE TRAIL: Tighten trail when signal turns bearish")
print("="*100)

# Compare fixed vs adaptive trail for both strategies
print("\nSTRAT-002:")
fixed_002 = backtest_strategy(df, 'strat002_entry', trail_pct=0.30)
adaptive_002 = backtest_adaptive_trail(df, 'strat002_entry', base_trail=0.30)
print(f"  Fixed 30% trail:    Return={fixed_002['total_return']:.0f}%, Sharpe={fixed_002['sharpe']:.2f}")
print(f"  Adaptive trail:     Return={adaptive_002['total_return']:.0f}%, Sharpe={adaptive_002['sharpe']:.2f}")

print("\nSTRAT-003:")
fixed_003 = backtest_strategy(df, 'strat003_entry', trail_pct=0.30, 
                               trigger_col='strat003_exit_trigger', tight_trail=0.15)
adaptive_003 = backtest_adaptive_trail(df, 'strat003_entry', base_trail=0.30)
print(f"  LTH-SOPR trigger:   Return={fixed_003['total_return']:.0f}%, Sharpe={fixed_003['sharpe']:.2f}")
print(f"  Adaptive trail:     Return={adaptive_003['total_return']:.0f}%, Sharpe={adaptive_003['sharpe']:.2f}")

---
## Part 5: Visualize Best Combinations

In [ ]:
# Find best from each strategy
best_002 = max(results_002, key=lambda x: x['sharpe'])
best_003 = max(results_003, key=lambda x: x['sharpe'])

# HODL baseline
hodl = 100000 * (1 + df['returns']).cumprod()
years = (df.index[-1] - df.index[0]).days / 365
hodl_cagr = ((hodl.iloc[-1] / 100000) ** (1/years) - 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# STRAT-002 comparison
baseline_002 = results_002[0]
axes[0].semilogy(hodl.index, hodl, 'orange', linewidth=2, alpha=0.5, label=f'HODL ({hodl_cagr:.0f}%)')
axes[0].semilogy(baseline_002['equity'].index, baseline_002['equity'], '#3b82f6', 
                 linewidth=1.5, label=f"STRAT-002 Baseline ({baseline_002['cagr']:.0f}%)")
axes[0].semilogy(best_002['equity'].index, best_002['equity'], '#22c55e', 
                 linewidth=2, label=f"STRAT-002 + {best_002['name']} ({best_002['cagr']:.0f}%)")
axes[0].set_ylabel('Equity ($)')
axes[0].set_title('STRAT-002: With vs Without Checkmate Signal', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# STRAT-003 comparison
baseline_003 = results_003[0]
axes[1].semilogy(hodl.index, hodl, 'orange', linewidth=2, alpha=0.5, label=f'HODL ({hodl_cagr:.0f}%)')
axes[1].semilogy(baseline_003['equity'].index, baseline_003['equity'], '#3b82f6', 
                 linewidth=1.5, label=f"STRAT-003 Baseline ({baseline_003['cagr']:.0f}%)")
axes[1].semilogy(best_003['equity'].index, best_003['equity'], '#22c55e', 
                 linewidth=2, label=f"STRAT-003 + {best_003['name']} ({best_003['cagr']:.0f}%)")
axes[1].set_ylabel('Equity ($)')
axes[1].set_xlabel('Date')
axes[1].set_title('STRAT-003: With vs Without Checkmate Signal', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 6: Analyze Entry Signal Quality

In [ ]:
# When our strategies trigger, what does Checkmate signal say?
print("\nENTRY SIGNAL ANALYSIS: Checkmate Signal at Entry Points")
print("="*80)

for days in [30, 60, 90]:
    df[f'fwd_{days}d'] = df['price'].shift(-days) / df['price'] - 1

# STRAT-002 entries
s002_entries = df[df['strat002_entry']].copy()
print(f"\nSTRAT-002 Entries ({len(s002_entries)} signals):")
print(f"  Checkmate signal at entry: {s002_entries['checkmate_signal'].mean():.2f} (avg)")
print(f"  Range: {s002_entries['checkmate_signal'].min():.2f} to {s002_entries['checkmate_signal'].max():.2f}")
print(f"  Signal < 0 (bullish): {(s002_entries['checkmate_signal'] < 0).sum()} / {len(s002_entries)}")

# Compare returns when signal confirms vs doesn't
if len(s002_entries) > 3:
    confirmed = s002_entries['checkmate_signal'] < 0
    print(f"\n  90d forward return:")
    print(f"    With confirmation (signal<0):    {s002_entries.loc[confirmed, 'fwd_90d'].mean()*100:+.1f}%")
    print(f"    Without confirmation (signal>=0): {s002_entries.loc[~confirmed, 'fwd_90d'].mean()*100:+.1f}%")

# STRAT-003 entries
s003_entries = df[df['strat003_entry']].copy()
print(f"\nSTRAT-003 Entries ({len(s003_entries)} signals):")
print(f"  Checkmate signal at entry: {s003_entries['checkmate_signal'].mean():.2f} (avg)")
print(f"  Range: {s003_entries['checkmate_signal'].min():.2f} to {s003_entries['checkmate_signal'].max():.2f}")
print(f"  Signal < 0 (bullish): {(s003_entries['checkmate_signal'] < 0).sum()} / {len(s003_entries)}")

if len(s003_entries) > 3:
    confirmed = s003_entries['checkmate_signal'] < 0
    print(f"\n  90d forward return:")
    print(f"    With confirmation (signal<0):    {s003_entries.loc[confirmed, 'fwd_90d'].mean()*100:+.1f}%")
    print(f"    Without confirmation (signal>=0): {s003_entries.loc[~confirmed, 'fwd_90d'].mean()*100:+.1f}%")

In [ ]:
# Visualize entries with Checkmate signal
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price with entries
axes[0].semilogy(df.index, df['price'], 'white', linewidth=1, alpha=0.7)

# STRAT-002 entries
s002_confirmed = df['strat002_entry'] & (df['checkmate_signal'] < 0)
s002_unconfirmed = df['strat002_entry'] & (df['checkmate_signal'] >= 0)
axes[0].scatter(df.index[s002_confirmed], df.loc[s002_confirmed, 'price'], 
                color='#22c55e', s=100, marker='^', label='STRAT-002 Confirmed', zorder=5)
axes[0].scatter(df.index[s002_unconfirmed], df.loc[s002_unconfirmed, 'price'], 
                color='#ef4444', s=100, marker='x', label='STRAT-002 Unconfirmed', zorder=5)

axes[0].set_ylabel('Price ($)')
axes[0].set_title('Entry Signals: Confirmed vs Unconfirmed by Checkmate Signal', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Checkmate signal
axes[1].plot(df.index, df['checkmate_signal'], 'white', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='#3b82f6', linestyle='--', linewidth=2)
axes[1].fill_between(df.index, -3, 0, alpha=0.1, color='#22c55e', label='Bullish zone')
axes[1].fill_between(df.index, 0, 3, alpha=0.1, color='#ef4444', label='Bearish zone')

# Mark entry points on signal
axes[1].scatter(df.index[df['strat002_entry']], df.loc[df['strat002_entry'], 'checkmate_signal'],
                color='yellow', s=50, marker='o', label='STRAT-002 Entry', zorder=5)

axes[1].set_ylabel('Checkmate Signal')
axes[1].set_xlabel('Date')
axes[1].set_ylim(-2.5, 2.5)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
print("\n" + "#"*80)
print("SUMMARY: CHECKMATE SIGNAL + OUR STRATEGIES")
print("#"*80)

print(f"""
STRAT-002 (SOPR + STH-SOPR + RL Z → 30% Trail):
  Baseline:           {results_002[0]['total_return']:.0f}% return, {results_002[0]['sharpe']:.2f} Sharpe
  Best w/ Checkmate:  {best_002['total_return']:.0f}% return, {best_002['sharpe']:.2f} Sharpe
  Best config:        {best_002['name']}
  Improvement:        {(best_002['sharpe'] - results_002[0]['sharpe']) / results_002[0]['sharpe'] * 100:+.1f}% Sharpe

STRAT-003 (STH-SOPR<1 → LTH-SOPR Triggered Trail):
  Baseline:           {results_003[0]['total_return']:.0f}% return, {results_003[0]['sharpe']:.2f} Sharpe
  Best w/ Checkmate:  {best_003['total_return']:.0f}% return, {best_003['sharpe']:.2f} Sharpe
  Best config:        {best_003['name']}
  Improvement:        {(best_003['sharpe'] - results_003[0]['sharpe']) / results_003[0]['sharpe'] * 100:+.1f}% Sharpe

RECOMMENDATIONS:
""")

# Determine best use
if best_002['sharpe'] > results_002[0]['sharpe']:
    print(f"  ✅ STRAT-002: Use Checkmate signal for {best_002['name']}")
else:
    print(f"  ❌ STRAT-002: Keep original (Checkmate doesn't help)")

if best_003['sharpe'] > results_003[0]['sharpe']:
    print(f"  ✅ STRAT-003: Use Checkmate signal for {best_003['name']}")
else:
    print(f"  ❌ STRAT-003: Keep original (Checkmate doesn't help)")

# Current state
latest = df.iloc[-1]
print(f"\nCURRENT STATE ({latest.name.date()}):")
print(f"  Price: ${latest['price']:,.0f}")
print(f"  Checkmate Signal: {latest['checkmate_signal']:+.2f}")
print(f"  SOPR: {latest['sopr']:.3f}")
print(f"  STH-SOPR: {latest['sth_sopr']:.3f}")
print(f"  MVRV: {latest['mvrv']:.2f}")

# Entry confirmation
print(f"\n  Entry confirmation (signal < 0): {'✅ YES' if latest['checkmate_signal'] < 0 else '❌ NO'}")